In [ ]:
from datetime import date

import numpy as np
import pandas as pd
import yaml
from sqlalchemy import create_engine
##preguntas que responde este hecho
######1.2.4.6.

##origen servicio como sede

##usuario pertenece a sede

##usuario pertenece a solo una sede

# database connections 

In [2]:
with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['CO_SA']
    config_etl = config['ETL_PRO']

url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
co_sa = create_engine(url_co)
etl_conn = create_engine(url_etl)

# Extract

In [ ]:
#hecho_servicio = pd.read_sql_table('mensajeria_servicio', url_co)
tabla_fases = pd.read_sql_table('mensajeria_estadosservicio', url_co)
dim_estado = pd.read_sql_table('dim_estado', etl_conn)
dim_fecha = pd.read_sql_table('dim_fecha', etl_conn)
dim_hora = pd.read_sql_table('dim_hora', etl_conn)
dim_geografia = pd.read_sql_table('dim_geografia', etl_conn)
dim_mensajero = pd.read_sql_table('dim_mensajero', etl_conn)
dim_novedad = pd.read_sql_table('dim_novedad', etl_conn)
dim_sede = pd.read_sql_table('dim_sede', etl_conn)
dim_tipo_servicio = pd.read_sql_table('dim_tipo_servicio', etl_conn)
dim_estado.info()
#select count(*) from mensajeria_documentoasociado; 


<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   key_dim_estado   6 non-null      int64  
 1   id               6 non-null      int64  
 2   nombre_estado    6 non-null      str    
 3   orden_secuencia  5 non-null      float64
dtypes: float64(1), int64(2), str(1)
memory usage: 324.0 bytes


# Transformations



In [23]:
hecho_servicio = pd.read_sql_table('mensajeria_servicio', url_co)

In [24]:
hecho_servicio.drop(columns=['descripcion','fecha_deseada','hora_deseada', 'nombre_recibe','telefono_recibe','descripcion_pago','ida_y_regreso','activo','novedades','tipo_pago_id','tipo_vehiculo_id','prioridad','hora_visto_por_mensajero','visto_por_mensajero','descripcion_multiples_origenes','mensajero2_id','mensajero3_id','multiples_origenes','asignar_mensajero','es_prueba','descripcion_cancelado'], inplace=True)

hecho_servicio.drop(columns=['destino_id','mensajero_id','origen_id','tipo_servicio_id','usuario_id','ciudad_destino_id','ciudad_origen_id'], inplace=True)
#hecho_servicio.drop(columns=['fecha_solicitud','hora_solicitud','nombre_recibe','cliente_id'], inplace=True)

hecho_servicio.columns.tolist()

['id', 'nombre_solicitante', 'fecha_solicitud', 'hora_solicitud', 'cliente_id']

In [34]:
hecho_servicio.info()

<class 'pandas.DataFrame'>
RangeIndex: 28430 entries, 0 to 28429
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype        
---  ------              --------------  -----        
 0   id_servicio         28430 non-null  int64        
 1   nombre_solicitante  28430 non-null  str          
 2   fecha_solicitud     28430 non-null  datetime64[s]
 3   hora_solicitud      28430 non-null  object       
 4   id_cliente          28430 non-null  int64        
dtypes: datetime64[s](1), int64(2), object(1), str(1)
memory usage: 1.1+ MB


# Alinear nombres de columnas al diagrama
ID_Estado (PK) -> indice al cargar | Nombre_Estado | Orden_Secuencia

In [26]:
hecho_servicio.rename(columns={
    'id': 'id_servicio','cliente_id': 'id_cliente'
}, inplace=True)

In [33]:
#hecho_servicio.head()
hecho_servicio['id_cliente'].unique()

array([ 5,  4,  2, 11,  7,  3,  6, 25,  9, 12, 24,  8, 22, 27])

# load

In [ ]:
# NOTA: cambiado de if_exists='replace' a 'append' porque la tabla ahora se crea por DDL
# (ver sqlscripts.yml) con su PK real. 'replace' borraria esa tabla y la reemplazaria sin
# restricciones. Antes de correr este notebook, asegurate que main.py ya ejecuto el DDL
# (o ejecuta el script de sqlscripts.yml manualmente si la tabla aun no existe).
hecho_servicio.to_sql('hecho_servicio', etl_conn, if_exists='replace', index_label='key_dim_estado')

6